### All MySQL commands using Python for automating MySQL

In [1]:
import mysql.connector

DB_CONFIG = dict(host="localhost", user="root", password="garvit@123", database="aegis_neo")

def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

def run(sql, description=""):
    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute(sql)
        conn.commit()
        print(f"✅ {description or 'OK'} | rows affected: {cur.rowcount}")
    except Exception as e:
        print(f"❌ {description}: {e}")
    finally:
        cur.close(); conn.close()

def fetch(sql):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    cur.close(); conn.close()
    return cols, rows

In [2]:
run("""
CREATE TABLE IF NOT EXISTS dim_neo (
    neo_id VARCHAR(20) PRIMARY KEY,
    full_name VARCHAR(150),
    is_hazardous BOOLEAN,
    diameter_km_avg FLOAT,
    absolute_magnitude_h FLOAT,
    eccentricity FLOAT,
    semi_major_axis_au FLOAT,
    inclination_deg FLOAT,
    orbital_period_days FLOAT,
    data_arc_days INT,
    n_observations INT,
    impact_probability FLOAT,
    palermo_scale_max FLOAT,
    torino_scale INT,
    last_obs_date VARCHAR(20)
)
""", "create dim_neo")

run("""
CREATE TABLE IF NOT EXISTS fact_close_approach (
    approach_id INT AUTO_INCREMENT PRIMARY KEY,
    neo_id VARCHAR(20),
    close_approach_date DATE,
    relative_velocity_kmh FLOAT,
    miss_distance_km FLOAT,
    miss_distance_ld FLOAT,
    orbiting_body VARCHAR(50),
    FOREIGN KEY (neo_id) REFERENCES dim_neo(neo_id)
)
""", "create fact_close_approach")

✅ create dim_neo | rows affected: 0
✅ create fact_close_approach | rows affected: 0


In [3]:
run("""
INSERT INTO dim_neo (neo_id, full_name, diameter_km_avg, absolute_magnitude_h,
    eccentricity, semi_major_axis_au, inclination_deg, orbital_period_days,
    data_arc_days, n_observations, impact_probability, palermo_scale_max,
    torino_scale, last_obs_date, is_hazardous)
SELECT
    o.neo_id, TRIM(o.full_name),
    AVG((f.est_diameter_min_km + f.est_diameter_max_km) / 2),
    NULL, o.eccentricity, o.semi_major_axis_au, o.inclination_deg,
    o.orbital_period_days, o.data_arc_days, o.n_observations,
    s.impact_probability, s.palermo_scale_max, s.torino_scale,
    s.last_obs_date, MAX(f.is_hazardous)
FROM raw_orbital_elements o
LEFT JOIN raw_sentry_risk s ON o.neo_id = s.neo_id
LEFT JOIN raw_neo_feed f ON o.neo_id = f.neo_id
GROUP BY o.neo_id, o.full_name, o.eccentricity, o.semi_major_axis_au,
         o.inclination_deg, o.orbital_period_days, o.data_arc_days,
         o.n_observations, s.impact_probability, s.palermo_scale_max,
         s.torino_scale, s.last_obs_date
""", "populate dim_neo")

❌ populate dim_neo: 1062 (23000): Duplicate entry '20000433' for key 'dim_neo.PRIMARY'


In [4]:
run("""
INSERT INTO fact_close_approach (neo_id, close_approach_date, relative_velocity_kmh,
    miss_distance_km, miss_distance_ld, orbiting_body)
SELECT f.neo_id, f.close_approach_date, f.relative_velocity_kmh, f.miss_distance_km,
    f.miss_distance_km / 384400, f.orbiting_body
FROM raw_neo_feed f
INNER JOIN dim_neo d ON f.neo_id = d.neo_id
""", "populate fact_close_approach")

✅ populate fact_close_approach | rows affected: 1


In [5]:
cols, rows = fetch("SELECT COUNT(*) FROM raw_neo_feed")
print("raw_neo_feed total:", rows[0][0])
cols, rows = fetch("SELECT COUNT(DISTINCT neo_id) FROM raw_neo_feed")
print("raw_neo_feed unique neo_id:", rows[0][0])
cols, rows = fetch("SELECT COUNT(*) FROM fact_close_approach")
print("fact_close_approach inserted:", rows[0][0])


raw_neo_feed total: 896
raw_neo_feed unique neo_id: 872
fact_close_approach inserted: 822


In [6]:
import pandas as pd

In [7]:
cols, rows = fetch("SELECT neo_id, name FROM raw_neo_feed LIMIT 5")
print(pd.DataFrame(rows, columns=cols))

cols, rows = fetch("SELECT neo_id, full_name FROM dim_neo LIMIT 5")
print(pd.DataFrame(rows, columns=cols))

    neo_id                    name
0  2001943  1943 Anteros (1973 EC)
1  2002340   2340 Hathor (1976 UA)
2  2017182         17182 (1999 VU)
3  2052340         52340 (1992 SY)
4  2054071      54071 (2000 GQ146)
     neo_id               full_name
0  20000433      433 Eros (A898 PA)
1  20000719    719 Albert (A911 TB)
2  20000887    887 Alinda (A918 AA)
3  20001036  1036 Ganymed (A924 UB)
4  20001221    1221 Amor (1932 EA1)


In [8]:
run("ALTER TABLE raw_neo_feed ADD COLUMN designation_num VARCHAR(20)", "add designation_num to raw_neo_feed")
run("ALTER TABLE dim_neo ADD COLUMN designation_num VARCHAR(20)", "add designation_num to dim_neo")

run("""
UPDATE raw_neo_feed
SET designation_num = TRIM(SUBSTRING_INDEX(TRIM(name), ' ', 1))
""", "populate designation_num in raw_neo_feed")

run("""
UPDATE dim_neo
SET designation_num = TRIM(SUBSTRING_INDEX(TRIM(full_name), ' ', 1))
""", "populate designation_num in dim_neo")

❌ add designation_num to raw_neo_feed: 1060 (42S21): Duplicate column name 'designation_num'
❌ add designation_num to dim_neo: 1060 (42S21): Duplicate column name 'designation_num'
✅ populate designation_num in raw_neo_feed | rows affected: 804
✅ populate designation_num in dim_neo | rows affected: 707


In [9]:
cols, rows = fetch("""
SELECT COUNT(*) FROM raw_neo_feed f
INNER JOIN dim_neo d ON f.designation_num = d.designation_num
""")
print("matched rows:", rows[0][0])

matched rows: 44480


In [10]:
import requests
import mysql.connector

API_KEY = "hBnrFPk07qCcnQy1unyYO2VDgD1nAKH9gi8liBHu"
DB_CONFIG = dict(host="localhost", user="root", password="garvit@123", database="aegis_neo")

def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

def fetch_neo_feed(start_date, end_date):
    url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date={start_date}&end_date={end_date}&api_key={API_KEY}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()["near_earth_objects"]

def insert_feed_data(data):
    conn = get_conn()
    cur = conn.cursor()
    for date, objects in data.items():
        for obj in objects:
            approach = obj["close_approach_data"][0]
            cur.execute("""
                INSERT IGNORE INTO raw_neo_feed
                (neo_id, name, absolute_magnitude_h, est_diameter_min_km, est_diameter_max_km,
                 is_hazardous, close_approach_date, relative_velocity_kmh, miss_distance_km, orbiting_body)
                VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
            """, (
                obj["id"], obj["name"], obj["absolute_magnitude_h"],
                obj["estimated_diameter"]["kilometers"]["estimated_diameter_min"],
                obj["estimated_diameter"]["kilometers"]["estimated_diameter_max"],
                obj["is_potentially_hazardous_asteroid"], date,
                float(approach["relative_velocity"]["kilometers_per_hour"]),
                float(approach["miss_distance"]["kilometers"]),
                approach["orbiting_body"]
            ))
    conn.commit()
    cur.close(); conn.close()

In [11]:
import time
from datetime import datetime, timedelta

today = datetime.today()
for i in range(0, 180, 7):  # ~6 months, week by week
    start = (today - timedelta(days=i+7)).strftime("%Y-%m-%d")
    end = (today - timedelta(days=i)).strftime("%Y-%m-%d")
    try:
        insert_feed_data(fetch_neo_feed(start, end))
        print(f"✅ {start} to {end}")
    except Exception as e:
        print(f"❌ {start} to {end}: {e}")
    time.sleep(1)

✅ 2026-09-04 to 2026-09-11
✅ 2026-08-28 to 2026-09-04
✅ 2026-08-21 to 2026-08-28
✅ 2026-08-14 to 2026-08-21
✅ 2026-08-07 to 2026-08-14
✅ 2026-07-31 to 2026-08-07
✅ 2026-07-24 to 2026-07-31
✅ 2026-07-17 to 2026-07-24
✅ 2026-07-10 to 2026-07-17
✅ 2026-07-03 to 2026-07-10
✅ 2026-06-26 to 2026-07-03
✅ 2026-06-19 to 2026-06-26
✅ 2026-06-12 to 2026-06-19
✅ 2026-06-05 to 2026-06-12
✅ 2026-05-29 to 2026-06-05
✅ 2026-05-22 to 2026-05-29
✅ 2026-05-15 to 2026-05-22
✅ 2026-05-08 to 2026-05-15
✅ 2026-05-01 to 2026-05-08
✅ 2026-04-24 to 2026-05-01
✅ 2026-04-17 to 2026-04-24
✅ 2026-04-10 to 2026-04-17
✅ 2026-04-03 to 2026-04-10
✅ 2026-03-27 to 2026-04-03
✅ 2026-03-20 to 2026-03-27
✅ 2026-03-13 to 2026-03-20


In [12]:
import requests

def fetch_orbital_by_designation(designation):
    url = "https://ssd-api.jpl.nasa.gov/sbdb.api"
    params = {"sstr": designation, "full-prec": "true"}
    r = requests.get(url, params=params, timeout=20)
    if r.status_code != 200:
        return None
    return r.json()

# get distinct designations from your (now much bigger) feed table
cols, rows = fetch("SELECT DISTINCT designation_num FROM raw_neo_feed")
designations = [r[0] for r in rows]
print(f"{len(designations)} unique objects to fetch")

122 unique objects to fetch


In [13]:
run("""
UPDATE raw_neo_feed
SET designation_num = TRIM(SUBSTRING_INDEX(TRIM(name), ' ', 1))
WHERE designation_num IS NULL
""", "backfill designation_num for new rows")

✅ backfill designation_num for new rows | rows affected: 252


In [14]:
cols, rows = fetch("SELECT COUNT(*) FROM raw_neo_feed")
print("total feed rows:", rows[0][0])

cols, rows = fetch("SELECT DISTINCT designation_num FROM raw_neo_feed")
designations = [r[0] for r in rows]
print(f"{len(designations)} unique objects to fetch")

total feed rows: 1148
141 unique objects to fetch


In [15]:
import requests
import time

def fetch_orbital_by_designation(designation):
    url = "https://ssd-api.jpl.nasa.gov/sbdb.api"
    params = {"sstr": designation, "full-prec": "true"}
    r = requests.get(url, params=params, timeout=20)
    if r.status_code != 200:
        return None
    return r.json()

results = {}
for i, des in enumerate(designations):
    try:
        data = fetch_orbital_by_designation(des)
        if data and "orbit" in data:
            results[des] = data
            print(f"✅ {i+1}/{len(designations)} {des}")
        else:
            print(f"⚠️ {i+1}/{len(designations)} {des} — no orbit data")
    except Exception as e:
        print(f"❌ {des}: {e}")
    time.sleep(0.5)  # respect rate limit

✅ 1/141 1620
✅ 2/141 1943
✅ 3/141 2340
✅ 4/141 17182
✅ 5/141 52340
✅ 6/141 54071
✅ 7/141 66146
✅ 8/141 88710
✅ 9/141 90416
✅ 10/141 136770
✅ 11/141 136818
✅ 12/141 141495
✅ 13/141 141531
✅ 14/141 152637
✅ 15/141 162882
✅ 16/141 164216
✅ 17/141 173561
✅ 18/141 186822
✅ 19/141 192559
✅ 20/141 221455
✅ 21/141 240320
✅ 22/141 242708
✅ 23/141 244670
✅ 24/141 246138
✅ 25/141 248590
✅ 26/141 250680
✅ 27/141 253062
✅ 28/141 276033
✅ 29/141 276891
✅ 30/141 302831
✅ 31/141 310842
✅ 32/141 318411
✅ 33/141 326290
✅ 34/141 330659
✅ 35/141 337075
✅ 36/141 357621
✅ 37/141 363599
✅ 38/141 367943
✅ 39/141 374038
✅ 40/141 375103
✅ 41/141 376879
✅ 42/141 388945
✅ 43/141 394392
✅ 44/141 398188
✅ 45/141 399325
✅ 46/141 427684
✅ 47/141 434196
✅ 48/141 437844
✅ 49/141 439877
✅ 50/141 441987
✅ 51/141 452639
✅ 52/141 454101
✅ 53/141 465824
✅ 54/141 467336
✅ 55/141 469219
✅ 56/141 470310
✅ 57/141 476187
✅ 58/141 478784
✅ 59/141 480858
✅ 60/141 485823
✅ 61/141 488615
✅ 62/141 494975
✅ 63/141 497626
✅ 64/141 4999

In [16]:
import re

def clean_designation(name):
    name = name.strip()
    if name.startswith("("):
        # provisional designation, e.g. "(2020 AB1)" -> "2020 AB1"
        return name.strip("()")
    else:
        # numbered asteroid, e.g. "240320 (2003 HS42)" -> "240320"
        return name.split(" ")[0]

# rebuild designations list correctly from raw_neo_feed
cols, rows = fetch("SELECT DISTINCT name FROM raw_neo_feed")
designations = [clean_designation(r[0]) for r in rows]
print(f"{len(designations)} unique objects to fetch")
print(designations[:10])

1092 unique objects to fetch
['1620', '1943', '2340', '17182', '52340', '54071', '66146', '88710', '90416', '136770']


In [17]:
run("""
UPDATE raw_neo_feed
SET designation_num = TRIM(TRAILING ')' FROM TRIM(LEADING '(' FROM TRIM(name)))
WHERE name LIKE '(%'
""", "fix provisional designation_num")

run("""
UPDATE raw_neo_feed
SET designation_num = TRIM(SUBSTRING_INDEX(TRIM(name), ' ', 1))
WHERE name NOT LIKE '(%'
""", "fix numbered designation_num")

✅ fix provisional designation_num | rows affected: 1034
✅ fix numbered designation_num | rows affected: 0


In [18]:
cols, rows = fetch("SELECT DISTINCT name FROM raw_neo_feed")
designations = [clean_designation(r[0]) for r in rows]
print(f"{len(designations)} unique objects to fetch")

1092 unique objects to fetch


In [ ]:
results = {}
for i, des in enumerate(designations):
    try:
        data = fetch_orbital_by_designation(des)
        if data and "orbit" in data:
            results[des] = data
            print(f"✅ {i+1}/{len(designations)} {des}")
        else:
            print(f"⚠️ {i+1}/{len(designations)} {des} — no orbit data")
    except Exception as e:
        print(f"❌ {des}: {e}")
    time.sleep(0.5)

✅ 1/1092 1620
✅ 2/1092 1943
✅ 3/1092 2340
✅ 4/1092 17182
✅ 5/1092 52340
✅ 6/1092 54071
✅ 7/1092 66146
✅ 8/1092 88710
✅ 9/1092 90416
✅ 10/1092 136770
✅ 11/1092 136818
✅ 12/1092 141495
✅ 13/1092 141531
✅ 14/1092 152637
✅ 15/1092 162882


In [ ]:
def insert_dim_neo_from_sbdb(results):
    conn = get_conn()
    cur = conn.cursor()
    for des, data in results.items():
        obj = data.get("object", {})
        orbit = data.get("orbit", {})
        elems = {e["name"]: e["value"] for e in orbit.get("elements", [])}
        spkid = obj.get("spkid")
        full_name = obj.get("fullname")

        cur.execute("""
            INSERT INTO dim_neo
            (neo_id, full_name, designation_num, eccentricity, semi_major_axis_au,
             inclination_deg, orbital_period_days, data_arc_days, n_observations)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
            ON DUPLICATE KEY UPDATE full_name=VALUES(full_name)
        """, (
            spkid, full_name, des,
            float(elems.get("e", 0) or 0),
            float(elems.get("a", 0) or 0),
            float(elems.get("i", 0) or 0),
            float(elems.get("per", 0) or 0),
            None, None
        ))
    conn.commit()
    cur.close(); conn.close()

insert_dim_neo_from_sbdb(results)

In [ ]:
insert_dim_neo_from_sbdb(results)

In [ ]:
cols, rows = fetch("SELECT COUNT(*) FROM dim_neo")
print("dim_neo rows:", rows[0][0])

cols, rows = fetch("""
SELECT COUNT(*) FROM raw_neo_feed f
INNER JOIN dim_neo d ON f.designation_num = d.designation_num
""")
print("matched rows for fact table:", rows[0][0])

dim_neo rows: 2760
matched rows for fact table: 821


In [ ]:
run("ALTER TABLE dim_neo ADD COLUMN priority_score FLOAT", "add priority_score column")

run("""
UPDATE dim_neo
SET priority_score = 
    (COALESCE(impact_probability,0) * 100000) +
    (CASE WHEN is_hazardous = 1 THEN 20 ELSE 0 END) +
    (CASE WHEN data_arc_days < 365 THEN 15 ELSE 0 END) +
    (CASE WHEN n_observations < 50 THEN 15 ELSE 0 END) +
    (CASE WHEN torino_scale > 0 THEN 30 ELSE 0 END)
""", "compute priority_score")

✅ add priority_score column | rows affected: 0
✅ compute priority_score | rows affected: 2760


In [ ]:
cols, rows = fetch("SELECT COUNT(*) FROM dim_neo")
print("dim_neo rows:", rows[0][0])

cols, rows = fetch("SELECT COUNT(*) FROM fact_close_approach")
print("fact_close_approach rows:", rows[0][0])

cols, rows = fetch("SELECT full_name, priority_score FROM dim_neo ORDER BY priority_score DESC LIMIT 10")
import pandas as pd
print(pd.DataFrame(rows, columns=cols))

dim_neo rows: 2760
fact_close_approach rows: 0
                   full_name  priority_score
0        447977 (2008 CC119)            15.0
1         434053 (2001 UP27)            15.0
2  367943 Duende (2012 DA14)            15.0
3         163015 (2001 UX16)            15.0
4       719 Albert (A911 TB)             0.0
5       887 Alinda (A918 AA)             0.0
6       (2026 EU3 = 2026 FM)             0.0
7       1221 Amor (1932 EA1)             0.0
8      1566 Icarus (1949 MA)             0.0
9  1620 Geographos (1951 RA)             0.0


In [ ]:
run("""
INSERT INTO fact_close_approach (neo_id, close_approach_date, relative_velocity_kmh,
    miss_distance_km, miss_distance_ld, orbiting_body)
SELECT d.neo_id, f.close_approach_date, f.relative_velocity_kmh, f.miss_distance_km,
    f.miss_distance_km / 384400, f.orbiting_body
FROM raw_neo_feed f
INNER JOIN dim_neo d ON f.designation_num = d.designation_num
""", "populate fact_close_approach")

✅ populate fact_close_approach | rows affected: 821


In [ ]:
cols, rows = fetch("SELECT COUNT(*) FROM fact_close_approach")
print("fact_close_approach rows:", rows[0][0])

fact_close_approach rows: 821


In [ ]:
run("""
UPDATE dim_neo d
INNER JOIN raw_neo_feed f ON d.designation_num = f.designation_num
SET d.is_hazardous = f.is_hazardous
WHERE d.is_hazardous IS NULL
""", "backfill is_hazardous")

✅ backfill is_hazardous | rows affected: 799


In [ ]:
run("""
UPDATE dim_neo
SET priority_score = 
    (COALESCE(impact_probability,0) * 100000) +
    (CASE WHEN is_hazardous = 1 THEN 20 ELSE 0 END) +
    (CASE WHEN data_arc_days < 365 THEN 15 ELSE 0 END) +
    (CASE WHEN n_observations < 50 THEN 15 ELSE 0 END) +
    (CASE WHEN torino_scale > 0 THEN 30 ELSE 0 END)
""", "recompute priority_score")

✅ recompute priority_score | rows affected: 78


In [ ]:
cols, rows = fetch("SELECT full_name, priority_score FROM dim_neo ORDER BY priority_score DESC LIMIT 10")
print(pd.DataFrame(rows, columns=cols))

               full_name  priority_score
0      152637 (1997 NC1)            20.0
1    276033 (2002 AJ129)            20.0
2       192559 (1998 VO)            20.0
3     162882 (2001 FD58)            20.0
4       302831 (2003 FH)            20.0
5      250680 (2005 QC5)            20.0
6     141495 (2002 EZ11)            20.0
7      242708 (2005 UK1)            20.0
8     90416 (2003 YK118)            20.0
9  2340 Hathor (1976 UA)            20.0
